# Hope: Transformer Model Training & Export

This notebook trains the Transformer-based probability model for the `hope` trading engine and exports it to ONNX format.

## 1. Setup Dependencies

In [ ]:
!pip install torch onnx onnxscript numpy

## 2. Define Model Architecture

In [ ]:
import torch
import torch.nn as nn
import torch.onnx
import numpy as np
import sqlite3

class SimpleTransformer(nn.Module):
    def __init__(self, input_dim=5, d_model=16, nhead=2, num_layers=2):
        super(SimpleTransformer, self).__init__()
        self.embedding = nn.Linear(input_dim, d_model)
        encoder_layers = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers=num_layers)
        self.fc = nn.Linear(d_model, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # x shape: (batch, seq_len, input_dim)
        x = self.embedding(x)
        x = self.transformer_encoder(x)
        # Use the last sequence element
        x = x[:, -1, :]
        x = self.fc(x)
        return self.sigmoid(x)

## 3. Data Preparation

Upload your `tick_store.db` to the Colab environment before running this cell.

In [ ]:
def load_real_data(db_path, limit=100000):
    print(f"Loading data from {db_path}...")
    try:
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()
        cursor.execute("SELECT quote FROM ticks ORDER BY epoch ASC LIMIT ?", (limit,))
        rows = cursor.fetchall()
        conn.close()
        return np.array([r[0] for r in rows], dtype=np.float32)
    except Exception as e:
        print(f"Error loading database: {e}")
        return None

def prepare_features(prices, seq_len=16):
    print("Preparing features...")
    returns = np.diff(prices)
    directions = np.sign(returns)
    magnitudes = np.abs(returns)
    
    streaks = []
    reversals = []
    curr_streak = 0
    curr_reversal = 0
    last_dir = 0
    
    for d in directions:
        if d == last_dir:
            curr_streak += 1
            curr_reversal += 1
        else:
            curr_streak = 1
            curr_reversal = 1
            last_dir = d
        streaks.append(curr_streak)
        reversals.append(curr_reversal)
        
    vol = []
    for i in range(len(returns)):
        start = max(0, i - 19)
        vol.append(np.std(returns[start:i+1]))
        
    features = np.stack([directions, magnitudes, streaks, reversals, vol], axis=1)
    
    x, y = [], []
    for i in range(len(features) - seq_len):
        x.append(features[i:i+seq_len])
        y.append(1.0 if returns[i+seq_len] > 0 else 0.0)
        
    return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.float32).unsqueeze(1)

## 4. Train Model

In [ ]:
db_path = "tick_store.db"
seq_len = 16
input_dim = 5

prices = load_real_data(db_path)
if prices is not None:
    x_train, y_train = prepare_features(prices, seq_len)
else:
    print("Using synthetic data as fallback...")
    x_train = torch.randn(1000, seq_len, input_dim)
    y_train = torch.rand(1000, 1)

model = SimpleTransformer(input_dim=input_dim)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.BCELoss()

print(f"Training on {len(x_train)} samples...")
batch_size = 64
for epoch in range(10):
    epoch_loss = 0
    for i in range(0, len(x_train), batch_size):
        batch_x = x_train[i:i+batch_size]
        batch_y = y_train[i:i+batch_size]
        
        optimizer.zero_grad()
        output = model(batch_x)
        loss = criterion(output, batch_y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        
    print(f"Epoch {epoch}, Loss: {epoch_loss / (len(x_train)/batch_size):.4f}")

## 5. Export to ONNX

In [ ]:
dummy_input = torch.randn(1, seq_len, input_dim)
onnx_path = "model.onnx"

print(f"Exporting to {onnx_path}...")
torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    export_params=True,
    opset_version=11,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['output']
)
print("Done! You can now download model.onnx from the file explorer.")

In [ ]:
from google.colab import files
files.download('model.onnx')